In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv


In [2]:
train = pd.read_csv('/kaggle/input/datasets/fedesoriano/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')

In [3]:
train.isnull().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [4]:
train.shape

(5110, 12)

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [6]:
train[['gender_Other','gender_Male']].value_counts()


KeyError: "None of [Index(['gender_Other', 'gender_Male'], dtype='object')] are in the [columns]"

In [ ]:
train = pd.get_dummies(train, columns=['gender','work_type','smoking_status'],
                       dtype=int,drop_first=True)

In [ ]:
train['ever_married'].value_counts()


In [ ]:
train['ever_married']= train['ever_married'].map({'Yes':1,'No':0})

In [ ]:
train[['work_type_Private','work_type_Self-employed','work_type_children',
      'work_type_Never_worked']].value_counts()

In [ ]:
train[['smoking_status_never smoked','smoking_status_formerly smoked',
       'smoking_status_smokes']].value_counts()

In [ ]:
train['Residence_type'].value_counts()

In [ ]:
train['Residence_type']=train['Residence_type'].map({'Urban':1,'Rural':0 })

In [ ]:
train['bmi']=train['bmi'].fillna(train['bmi'].median())

In [ ]:
y = train['stroke']

In [ ]:
X = train.drop(columns=['stroke','id'])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score,\
f1_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import GridSearchCV

In [ ]:
xgb = XGBClassifier(
    scale_pos_weight=4861/249,
    random_state=42,
    n_jobs=-1
)
cv_params = {'max_depth': [4,5,6,7,8], 
             'min_child_weight': [1,2,3,4,5],
             'learning_rate': [0.1, 0.2, 0.3],
             'n_estimators': [75, 100, 125]
             }    

scoring = ['accuracy', 'precision', 'recall', 'f1']
xgb_cv = GridSearchCV(xgb, cv_params, scoring=scoring, cv=5, refit='f1',n_jobs=-1)
xgb_cv.fit(X_train, y_train)
print("Best_param:", xgb_cv.best_params_)
print("Accuracy(Validation):", xgb_cv.best_score_)

In [ ]:
y_pred=xgb_cv.best_estimator_.predict(X_test)
# y_prob = xgb_cv.best_estimator_.predict_proba(X_test)[:, 1]
# y_pred = xgb_cv.best_estimator_.predict(X_test)
# y_pred = (y_prob >= 0.3).astype(int)
accuracy_score(y_pred, y_test)

In [ ]:
print("y_val:")
print(y.value_counts())

print("\ny_pred:")
print(pd.Series(y).value_counts())

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))